In [1]:
!nvidia-smi

Sun Aug 23 12:41:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls -lah "/content/drive/MyDrive/AquaGuard-SIH26057/data"

total 3.6M
-rw------- 1 root root  229 Aug 23 08:28 data.yaml
drwx------ 2 root root 4.0K Aug 23 08:28 images
drwx------ 2 root root 4.0K Aug 23 08:39 labels
-rw------- 1 root root 1.6M Aug 23 08:37 test_sample.jpg
-rw------- 1 root root 2.0M Aug 23 08:37 test_sample.png


In [4]:
!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/images" -type f | wc -l

6356


In [5]:
!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/labels" -type f | wc -l

6299


In [6]:
!ls -lah "/content/drive/MyDrive/AquaGuard-SIH26057/data"

total 3.6M
-rw------- 1 root root  229 Aug 23 08:28 data.yaml
drwx------ 5 root root 4.0K Aug 23 08:28 images
drwx------ 5 root root 4.0K Aug 23 08:39 labels
-rw------- 1 root root 1.6M Aug 23 08:37 test_sample.jpg
-rw------- 1 root root 2.0M Aug 23 08:37 test_sample.png


In [7]:
from pathlib import Path

root = Path("/content/drive/MyDrive/AquaGuard-SIH26057/data")

for split in ["train", "val", "test"]:
    images = list((root / "images" / split).glob("*"))
    labels = list((root / "labels" / split).glob("*.txt"))
    print(f"{split}: images={len(images)}, labels={len(labels)}")

train: images=5346, labels=5346
val: images=890, labels=890
test: images=120, labels=60


In [8]:
from pathlib import Path

root = Path("/content/drive/MyDrive/AquaGuard-SIH26057/data")

for split in ["train", "val", "test"]:
    image_stems = {p.stem for p in (root / "images" / split).iterdir() if p.is_file()}
    label_stems = {p.stem for p in (root / "labels" / split).glob("*.txt")}

    print(f"\n{split}")
    print("Images without labels:", len(image_stems - label_stems))
    print("Labels without images:", len(label_stems - image_stems))


train
Images without labels: 0
Labels without images: 0

val
Images without labels: 0
Labels without images: 0

test
Images without labels: 0
Labels without images: 0


In [10]:
!mkdir -p /content/AquaGuard/scripts

In [11]:
%cd /content/AquaGuard
!pip install -r requirements-colab.txt

/content/AquaGuard
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 7.7 MB/s eta 0:00:00


In [12]:
import torch
import ultralytics

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Ultralytics:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Ultralytics: 8.4.126


In [13]:
!rm -rf /content/AquaGuard/data
!mkdir -p /content/AquaGuard/data

!cp -r "/content/drive/MyDrive/AquaGuard-SIH26057/data/images" "/content/AquaGuard/data/"
!cp -r "/content/drive/MyDrive/AquaGuard-SIH26057/data/labels" "/content/AquaGuard/data/"
!cp "/content/drive/MyDrive/AquaGuard-SIH26057/data/data.yaml" "/content/AquaGuard/data/"

cp: error reading '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_HF_1693578514.580.pbm': Transport endpoint is not connected
cp: failed to close '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_HF_1693578514.580.pbm': Transport endpoint is not connected
cp: cannot stat '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_HF_1693578515.580.pbm': Transport endpoint is not connected
cp: cannot stat '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_HF_1693578516.580.pbm': Transport endpoint is not connected
cp: cannot stat '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_HF_1693578517.580.pbm': Transport endpoint is not connected
cp: cannot stat '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_HF_1693578518.580.pbm': Transport endpoint is not connected
cp: cannot stat '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_HF_1693578519.580.pbm': Transport endpoint is 

In [14]:
from pathlib import Path

root = Path("/content/AquaGuard/data")

for split in ["train", "val", "test"]:
    images = list((root / "images" / split).glob("*.pbm"))
    labels = list((root / "labels" / split).glob("*.txt"))
    print(f"{split}: images={len(images)}, labels={len(labels)}")

train: images=394, labels=0
val: images=0, labels=0
test: images=0, labels=0


In [15]:
!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train" -type f | wc -l
!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/images/val" -type f | wc -l
!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/images/test" -type f | wc -l

!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/labels/train" -type f | wc -l
!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/labels/val" -type f | wc -l
!find "/content/drive/MyDrive/AquaGuard-SIH26057/data/labels/test" -type f | wc -l

5346
890
120
5346
890
60


In [16]:
!ls -lh "/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_LF_1693575429.879.pbm"

ls: cannot access '/content/drive/MyDrive/AquaGuard-SIH26057/data/images/train/Chunk4_LF_1693575429.879.pbm': No such file or directory


In [17]:
!rm -rf /content/AquaGuard/data

In [18]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive/AquaGuard-SIH26057/data")

for split in ["train", "val", "test"]:
    files = list((drive_root / "images" / split).glob("*.pbm"))
    print(split, "PBM images:", len(files))
    print("First 3:")
    for p in files[:3]:
        print(" ", p.name)

train PBM images: 5346
First 3:
  Chunk4_HF_1693578460.569.pbm
  Chunk4_HF_1693578461.569.pbm
  Chunk4_HF_1693578462.569.pbm
val PBM images: 890
First 3:
  Chunk2_HF_1693573940.93.pbm
  Chunk2_HF_1693573941.93.pbm
  Chunk2_HF_1693573942.93.pbm
test PBM images: 60
First 3:
  Chunk3_HF_1693574476.03.pbm
  Chunk3_HF_1693574477.03.pbm
  Chunk3_HF_1693574478.03.pbm


In [19]:
!mkdir -p /content/AquaGuard/data/images /content/AquaGuard/data/labels

!rsync -ah --info=progress2 \
  "/content/drive/MyDrive/AquaGuard-SIH26057/data/images/" \
  "/content/AquaGuard/data/images/"

!rsync -ah --info=progress2 \
  "/content/drive/MyDrive/AquaGuard-SIH26057/data/labels/" \
  "/content/AquaGuard/data/labels/"

!cp "/content/drive/MyDrive/AquaGuard-SIH26057/data/data.yaml" \
  "/content/AquaGuard/data/data.yaml"

         35.55G 100%    7.78MB/s    1:12:36 (xfr#6356, to-chk=0/6360)
        454.58K 100%    0.16kB/s    0:43:32 (xfr#6299, to-chk=0/6303)


In [20]:
from pathlib import Path

root = Path("/content/AquaGuard/data")

for split in ["train", "val", "test"]:
    images = list((root / "images" / split).glob("*.pbm"))
    labels = list((root / "labels" / split).glob("*.txt"))
    print(f"{split}: images={len(images)}, labels={len(labels)}")

train: images=5346, labels=5346
val: images=890, labels=890
test: images=60, labels=60


In [21]:
%cd /content/AquaGuard

!python scripts/train_yolo.py \
    --model yolov8n.pt \
    --epochs 50 \
    --batch 16 \
    --imgsz 640 \
    --device 0 \
    --name baseline_yolov8n

/content/AquaGuard
AquaGuard — Baseline YOLO Pipeline
Project Root:    /content/AquaGuard
Model:           yolov8n.pt
Dataset config:  /content/AquaGuard/data/data.yaml
Project output:  /content/AquaGuard/models
Experiment name: baseline_yolov8n
Smoke test mode: False


[1/4] Loading model 'yolov8n.pt'...

[2/4] Starting training (epochs=50, batch=16, imgsz=640, fraction=1.0, device=0)...
Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/AquaGuard/data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed

In [22]:
%cd /content/AquaGuard

# Remove the interrupted YOLOv8n run/checkpoints
!rm -rf /content/AquaGuard/models/baseline_yolov8n
!rm -rf /content/AquaGuard/models/baseline_yolov8n_val
!rm -rf /content/AquaGuard/outputs/sample_predictions

# Start the real YOLOv8s training
!python scripts/train_yolo.py \
    --model yolov8s.pt \
    --epochs 50 \
    --batch 8 \
    --imgsz 640 \
    --device 0 \
    --name baseline_yolov8s

/content/AquaGuard
AquaGuard — Baseline YOLO Pipeline
Project Root:    /content/AquaGuard
Model:           yolov8s.pt
Dataset config:  /content/AquaGuard/data/data.yaml
Project output:  /content/AquaGuard/models
Experiment name: baseline_yolov8s
Smoke test mode: False


[1/4] Loading model 'yolov8s.pt'...

[2/4] Starting training (epochs=50, batch=8, imgsz=640, fraction=1.0, device=0)...
Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/AquaGuard/data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=N

In [23]:
%cd /content/AquaGuard

!rm -rf /content/AquaGuard/models/baseline_yolov8s
!rm -rf /content/AquaGuard/models/baseline_yolov8s_val
!rm -rf /content/AquaGuard/models/baseline_yolov8n

!python scripts/train_yolo.py \
    --model yolov8n.pt \
    --epochs 30 \
    --batch 16 \
    --imgsz 640 \
    --device 0 \
    --name baseline_yolov8n

/content/AquaGuard
AquaGuard — Baseline YOLO Pipeline
Project Root:    /content/AquaGuard
Model:           yolov8n.pt
Dataset config:  /content/AquaGuard/data/data.yaml
Project output:  /content/AquaGuard/models
Experiment name: baseline_yolov8n
Smoke test mode: False


[1/4] Loading model 'yolov8n.pt'...

[2/4] Starting training (epochs=30, batch=16, imgsz=640, fraction=1.0, device=0)...
Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/AquaGuard/data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed

In [24]:
from pathlib import Path
import shutil

src = Path("/content/AquaGuard/models/baseline_yolov8n")
dst = Path("/content/drive/MyDrive/AquaGuard-SIH26057/trained_model")

dst.mkdir(parents=True, exist_ok=True)

# Save the important checkpoints and training results
for name in [
    "weights/best.pt",
    "weights/last.pt",
    "results.csv",
    "args.yaml",
    "labels.jpg",
    "results.png",
]:
    source = src / name
    if source.exists():
        target = dst / source.name
        shutil.copy2(source, target)
        print("Saved:", target)

print("\nBackup complete.")

Saved: /content/drive/MyDrive/AquaGuard-SIH26057/trained_model/best.pt
Saved: /content/drive/MyDrive/AquaGuard-SIH26057/trained_model/last.pt
Saved: /content/drive/MyDrive/AquaGuard-SIH26057/trained_model/results.csv
Saved: /content/drive/MyDrive/AquaGuard-SIH26057/trained_model/args.yaml
Saved: /content/drive/MyDrive/AquaGuard-SIH26057/trained_model/labels.jpg
Saved: /content/drive/MyDrive/AquaGuard-SIH26057/trained_model/results.png

Backup complete.


In [25]:
!ls -lh "/content/drive/MyDrive/AquaGuard-SIH26057/trained_model"

total 13M
-rw------- 1 root root 1.7K Aug 23 15:33 args.yaml
-rw------- 1 root root 6.0M Aug 23 17:10 best.pt
-rw------- 1 root root  88K Aug 23 15:33 labels.jpg
-rw------- 1 root root 6.0M Aug 23 17:10 last.pt
-rw------- 1 root root 3.7K Aug 23 17:10 results.csv
-rw------- 1 root root 334K Aug 23 17:10 results.png


In [26]:
from ultralytics import YOLO

model = YOLO("/content/AquaGuard/models/baseline_yolov8n/weights/best.pt")

test_metrics = model.val(
    data="/content/AquaGuard/data/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
)

print("\n=== TEST RESULTS ===")
print(f"Precision : {test_metrics.box.mp:.4f}")
print(f"Recall    : {test_metrics.box.mr:.4f}")
print(f"mAP50     : {test_metrics.box.map50:.4f}")
print(f"mAP50-95  : {test_metrics.box.map:.4f}")

Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 185.1±33.9 MB/s, size: 1014.7 KB)
val: Scanning /content/AquaGuard/data/labels/test... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 487.2it/s 0.1s
val: New cache created: /content/AquaGuard/data/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.2it/s 3.2s
                   all         60         60      0.781      0.767       0.75       0.14
Speed: 4.4ms preprocess, 23.4ms inference, 0.0ms loss, 4.1ms postprocess per image
Results saved to /content/AquaGuard/runs/detect/val

=== TEST RESULTS ===
Precision : 0.7811
Recall    : 0.7667
mAP50     : 0.7503
mAP50-95  : 0.1404


In [27]:
from pathlib import Path
import shutil

drive = Path("/content/drive/MyDrive/AquaGuard-SIH26057/final_results")
drive.mkdir(parents=True, exist_ok=True)

# Model
shutil.copy2(
    "/content/AquaGuard/models/baseline_yolov8n/weights/best.pt",
    drive / "best.pt"
)

# Training metrics/results
for src in [
    "/content/AquaGuard/models/baseline_yolov8n/results.csv",
    "/content/AquaGuard/outputs/baseline_metrics.json",
]:
    p = Path(src)
    if p.exists():
        shutil.copy2(p, drive / p.name)

print("Final model and metrics backed up to:")
print(drive)

Final model and metrics backed up to:
/content/drive/MyDrive/AquaGuard-SIH26057/final_results


In [28]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("/content/AquaGuard/models/baseline_yolov8n/weights/best.pt")

test_images = list(
    Path("/content/AquaGuard/data/images/test").glob("*.pbm")
)[:5]

results = model.predict(
    source=[str(p) for p in test_images],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project="/content/AquaGuard/demo_predictions",
    name="sample"
)

print("Generated", len(results), "demo predictions.")


0: 640x640 1 Pipeline, 5.2ms
1: 640x640 2 Pipelines, 5.2ms
2: 640x640 1 Pipeline, 5.2ms
3: 640x640 1 Pipeline, 5.2ms
4: 640x640 3 Pipelines, 5.2ms
Speed: 1.9ms preprocess, 5.2ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/AquaGuard/demo_predictions/sample
Generated 5 demo predictions.


In [29]:
!cp -r /content/AquaGuard/demo_predictions/sample \
"/content/drive/MyDrive/AquaGuard-SIH26057/final_results/demo_predictions"

In [30]:
from pathlib import Path
import shutil

src = Path("/content/AquaGuard/models/baseline_yolov8n/weights/best.pt")
dst = Path("/content/drive/MyDrive/AquaGuard-SIH26057/final_results")
dst.mkdir(parents=True, exist_ok=True)

shutil.copy2(src, dst / "best.pt")
print("Saved:", dst / "best.pt")

Saved: /content/drive/MyDrive/AquaGuard-SIH26057/final_results/best.pt


In [31]:
!ls -lh "/content/drive/MyDrive/AquaGuard-SIH26057/final_results"

total 6.0M
-rw------- 1 root root  498 Aug 23 17:11 baseline_metrics.json
-rw------- 1 root root 6.0M Aug 23 17:10 best.pt
drwx------ 2 root root 4.0K Aug 23 17:14 demo_predictions
-rw------- 1 root root 3.7K Aug 23 17:10 results.csv


In [32]:
!cp -r /content/AquaGuard/demo_predictions/sample \
"/content/drive/MyDrive/AquaGuard-SIH26057/final_results/demo_predictions"